# 강의 03 · 실습 4 — RAG 에이전트 서비스 · (1) 강사 시연

## 1. 문제상황

- 구름월드 고객센터의 안내 프로그램은 구름월드 FAQ 검색기를 붙였는데도, FAQ에 없는 내용을 아는 척 답하는 일이 있습니다.
- 검색을 하고 안 하고를 사람이 정해 줄 수 없으니, 프로그램이 질문을 보고 사실 질문이면 FAQ를 찾아보고 인사말이면 바로 답해야 합니다.
- 근거가 없으면 지어내지 말고 정해진 문장으로 모른다고 답해야 합니다.
- 홈페이지 팀과 앱 팀은 이 안내 담당자를 각자의 화면에서 주소 하나로 불러 쓰고 싶어 합니다.

## 2. 문제와 목표

- **문제**: 검색할지 말지를 모델이 스스로 정하지 못하고, 근거가 없을 때 답을 지어내며, 다른 팀이 프로그램을 불러 쓸 주소가 없습니다.
- **목표**: 검색기를 도구로 등록해 모델이 사실 질문일 때만 검색하고, 근거가 없으면 고정 안내 문장으로 답하며, 재시도 상한이 있는 안내 담당자를 만들고, FastAPI 서비스의 `POST /ask` 주소 하나로 노출합니다.
    - 검색 도구: faq_search — 임계값 1.5 안의 청크만 돌려주고, 없으면 「검색 결과 없음」 문장을 돌려줍니다.
    - 고정 안내 문장: 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」
    - 안내 담당자: run_rag_agent — 시스템 프롬프트와 도구를 묶은 손 루프, 반복 상한 5.
    - 서비스: FastAPI 앱의 `POST /ask`(질문을 받아 답을 돌려줌)와 `GET /healthz`.
- **목표 달성 여부의 판정 기준**: 인사말은 검색 0회로 즉답하고, 환불 질문은 검색 1회 뒤 FAQ 근거로 답하며, 문서 밖 질문(파이썬 정렬)은 검색 뒤 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」로 끝나는 것을 노트북 실행과 서비스 호출(`/ask` 응답 JSON) 양쪽에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex04_s1_diagram.svg)

## 4. 단계별 요구사항

1. **문서를 적재합니다.**
    - `day05_faq_구름월드.csv`(32행)를 `csv.DictReader`로 읽고(`utf-8-sig`), `Question`이 빈 행은 버립니다.
    - 행마다 `[카테고리] Q: 질문\nA: 답변` 형식의 본문과 `{"row": 행 번호, "category": 카테고리}` 메타데이터를 가진 `Document`를 만들어 리스트 `docs`에 모으고, 적재 문서 수를 출력합니다.
2. **임베딩을 준비합니다.**
    - `OpenAIEmbeddings(model="text-embedding-3-small")`로 임베딩 부품 `emb`를 만듭니다.
3. **저장소를 구축하고 영속합니다.**
    - `Chroma.from_documents(docs, emb, persist_directory="chroma_db", ids=[...])`로 저장소 `db`를 만들고, 문서 id는 `row-<행 번호>`로 주어 셀을 다시 실행해도 항목이 늘지 않게 합니다.
4. **점수와 함께 검색합니다.**
    - `retrieve(query, k=2)`로 청크·점수 목록을 돌려주고, 「자유이용권 환불 규정」으로 시험해 점수를 출력합니다.
5. **임계값 컷을 품은 검색 도구를 선언합니다.**
    - `THRESHOLD = 1.5`. `@tool(parse_docstring=True)`를 붙인 `faq_search(query)`는 `retrieve`로 얻은 청크 중 점수가 임계값 이하인 것만 `[score …] 본문` 형식으로 이어 돌려주고, 하나도 없으면 「검색 결과 없음 (최고 유사도 점수 X가 임계값 1.5를 넘음). 질의를 바꿔 다시 검색하거나, 모른다고 답하라.」를 돌려줍니다.
    - 독스트링에 도구 설명(「시설 FAQ 문서에서 질문과 관련된 청크를 검색한다. 시설 이용, 요금, 환불, 운영 관련 질문에 쓴다.」)과 `query` 인자 설명을 적습니다.
6. **입출력 모양과 고정 문장을 선언합니다.**
    - `AskIn(question: str)`, `AskOut(answer: str)`, 고정 안내 문장 `NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."`, 시스템 프롬프트 `SYSTEM`(인사말은 즉답, 그 밖의 질문은 반드시 `faq_search`로 근거를 먼저 찾고 도구 결과로만 답하며, 도구 결과가 검색 결과 없음이면 `NO_EVIDENCE`로만 답한다)을 둡니다.
7. **처리 함수를 만듭니다.**
    - `run_rag_agent(question, max_turn=5)`는 시스템 프롬프트와 질문으로 대화 기록을 만들고, `llm.bind_tools([faq_search])` 손 루프를 돕니다.
    - 도구 호출이 없으면 답을 돌려주되, 검색을 했는데 근거가 한 번도 없었으면(`searched and not found`) `NO_EVIDENCE`를 돌려줍니다.
    - 검색 호출과 결과를 출력하고, 상한에 닿으면 「반복 한도 초과」를 돌려줍니다.
    - 인사말·환불·파이썬 세 질문으로 시험합니다.
8. **앱과 엔드포인트를 등록합니다.**
    - `%%writefile app.py`로 서비스 파일을 만듭니다.
    - 파일은 `.env`를 읽고 `chroma_db`를 다시 열어 4~7번의 도구·처리 함수를 그대로 담고, `app = FastAPI()`, `GET /healthz`, `POST /ask`(`AskIn`을 받아 `AskOut(answer=run_rag_agent(...))`를 돌려줌)를 등록합니다.
    - 서비스 표준 템플릿 `service_template\app.py`와 같은 구성입니다.
9. **기동하고 호출을 확인합니다.**
    - `fastapi dev app.py --port 8031 --no-reload`를 `subprocess`로 띄우고 `/healthz`가 200을 줄 때까지 기다린 뒤, `samples.json`의 질문 세 개를 `httpx.post("/ask")`로 보내 상태 코드와 응답 JSON을 출력하고, 서버를 종료합니다.
    - 터미널에서는 `fastapi dev app.py`와 `python run_samples.py 8031`로 같은 확인을 합니다.
    - 처리 함수는 도구를 부를 때 「[검색 호출] 인자 → 결과 앞부분」 줄을, 서버 확인은 「[healthz] 200 ok」 줄을 출력합니다.

## 5. 코드 골격 — RAG 인덱싱·검색 5단 + FastAPI 서비스 4단

두 골격을 잇습니다. RAG 다섯 단계 중 ⑤가 검색 도구 안으로 접힙니다. 서비스 네 단계는 서비스 표준 템플릿의 순서 그대로입니다. 「6. 코드 — 스텝바이스텝」의 코드 셀은 ①~⑤ 다음에 서비스 ①~④가 이어집니다.

**RAG 인덱싱·검색 5단** (검색 도구 안에 접힙니다)

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 문서 적재 | 원본 파일을 읽어 검색 단위 문서로 만듭니다 | `Document(page_content=..., metadata=...)` | 1 |
| ② 임베딩 준비 | 문장을 숫자 벡터로 바꿀 모델을 지정합니다 | `OpenAIEmbeddings(model="text-embedding-3-small")` | 2 |
| ③ 저장소 구축·영속 | 문서와 임베딩을 넣어 저장소를 만들고 디렉터리에 남깁니다 | `Chroma.from_documents(docs, emb, persist_directory=...)` | 3 |
| ④ 점수 동반 검색 | 질문을 넣어 가까운 문서와 그 거리 점수를 함께 받습니다 | `db.similarity_search_with_score(q, k=2)` | 4 |
| ⑤ 임계값 컷 | 점수가 기준을 넘으면 문서 근거를 쓰지 않고 다른 경로로 보냅니다 | `@tool faq_search` 안의 `if s <= THRESHOLD` | 5 |

**FastAPI 서비스 4단** (서비스 표준 템플릿 `service_template`의 구성과 같습니다)

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| 서비스 ① 입출력 모양 선언 | 서비스가 받을 값과 돌려줄 값의 모양을 클래스로 선언합니다 | `class AskIn(BaseModel)`, `class AskOut(BaseModel)` | 6 |
| 서비스 ② 처리 함수 구현 | 값을 받아 결과를 돌려주는 함수를 웹과 무관하게 먼저 만듭니다 | `def run_rag_agent(question) -> str` (bind_tools 손 루프) | 7 |
| 서비스 ③ 앱·엔드포인트 등록 | 앱을 만들고, 주소와 함수를 데코레이터로 잇습니다 | `app = FastAPI()`, `@app.post("/ask")` | 8 |
| 서비스 ④ 기동·호출 확인 | 개발 서버를 띄우고 요청을 보내 응답을 확인합니다 | `fastapi dev app.py --port ...`, `httpx.post("/ask")` | 9 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 읽습니다.

- API 키는 `.env` 파일에서 읽습니다. 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, `OPENAI_API_KEY=발급받은_키` 한 줄만 넣습니다.
- 서비스 파일 `app.py`도 같은 `.env`를 `find_dotenv(usecwd=True)`로 읽습니다.

In [ ]:
import csv
import json
import os
import subprocess
import sys
import time

import httpx
from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from pydantic import BaseModel

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

### 단계 ① — 문서 적재 (요구사항 1)

- FAQ 한 행을 청크 하나로 삼고, 질문과 답을 한 본문에 넣습니다.

In [ ]:
CSV_PATH = "day05_faq_구름월드.csv"

docs = []
with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    for i, row in enumerate(csv.DictReader(f), start=1):
        if not (row.get("Question") or "").strip():
            continue
        text = f"[{row['Category']}] Q: {row['Question']}\nA: {row['Answer']}"
        docs.append(Document(page_content=text, metadata={"row": i, "category": row["Category"]}))
print("적재 문서 수:", len(docs))

### 단계 ② — 임베딩 준비 (요구사항 2)

- 문장을 숫자 벡터로 바꿀 임베딩 모델을 지정합니다.

In [ ]:
emb = OpenAIEmbeddings(model="text-embedding-3-small")
print("임베딩 준비를 마쳤습니다.")

### 단계 ③ — 저장소 구축·영속 (요구사항 3)

- 저장소를 디렉터리에 남깁니다. 서비스 파일 `app.py`는 이 디렉터리를 다시 열어 씁니다.

In [ ]:
db = Chroma.from_documents(
    docs, emb,
    persist_directory="chroma_db",
    ids=[f"row-{d.metadata['row']}" for d in docs],
)
print("저장된 항목 수:", len(db.get()["ids"]))

### 단계 ④ — 점수 동반 검색 (요구사항 4)

- 점수 동반 검색을 함수 `retrieve`로 감쌉니다. 도구는 이 함수를 부릅니다.

In [ ]:
def retrieve(query: str, k: int = 2):
    """청크와 거리 점수의 목록을 돌려준다."""
    return db.similarity_search_with_score(query, k=k)


for d, s in retrieve("자유이용권 환불 규정"):
    print(f"  score={s:.4f} | {d.page_content[:50].replace(chr(10), ' / ')}")

### 단계 ⑤ — 임계값 컷을 품은 검색 도구 (요구사항 5)

- 임계값 컷이 도구 안에 들어갑니다. 도구는 임계값 안의 청크만 돌려주고, 없으면 「검색 결과 없음」 문장을 돌려줍니다. 이 문장이 모델에게 되먹여집니다.
- 독스트링이 도구 설명입니다. 모델은 이 설명을 보고 사실 질문에 도구를 부릅니다.

In [ ]:
THRESHOLD = 1.5


@tool(parse_docstring=True)
def faq_search(query: str) -> str:
    """시설 FAQ 문서에서 질문과 관련된 청크를 검색한다. 시설 이용, 요금, 환불, 운영 관련 질문에 쓴다.

    Args:
        query: 검색할 질의문. 결과가 없으면 표현을 바꿔 재검색할 수 있다.
    """
    hits = retrieve(query)
    good = [(d, s) for d, s in hits if s <= THRESHOLD]
    if not good:
        return f"검색 결과 없음 (최고 유사도 점수 {hits[0][1]:.2f}가 임계값 {THRESHOLD}를 넘음). 질의를 바꿔 다시 검색하거나, 모른다고 답하라."
    return "\n---\n".join(f"[score {s:.2f}] {d.page_content}" for d, s in good)


print("[도구]", faq_search.name, "|", faq_search.description)
print(faq_search.invoke({"query": "파이썬 리스트 정렬"})[:60])

### 서비스 단계 ① — 입출력 모양 선언 (요구사항 6)

- 요청 본문과 응답 본문의 모양을 pydantic 클래스로 못 박습니다. 고정 안내 문장과 시스템 프롬프트도 여기서 선언합니다.
- 시스템 프롬프트는 매 호출 그대로 들어가며, 검색할지 말지와 근거 없을 때의 태도를 정합니다.

In [ ]:
class AskIn(BaseModel):
    question: str


class AskOut(BaseModel):
    answer: str


NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."
SYSTEM = ("너는 시설 안내 담당자다. 인사말처럼 검색이 필요 없는 말에는 바로 답한다. "
          "그 밖의 모든 질문은 반드시 faq_search 도구로 근거를 먼저 찾고, 도구 결과에 있는 내용으로만 답한다. "
          "도구 결과가 '검색 결과 없음'이면 네가 아는 지식으로 답하지 말고 "
          f"'{NO_EVIDENCE}'라고만 답한다.")
print("요청 칸:", list(AskIn.model_fields), "| 응답 칸:", list(AskOut.model_fields))

### 서비스 단계 ② — 처리 함수 구현 (요구사항 7)

- 웹과 무관한 파이썬 함수로 먼저 만들고 노트북에서 시험합니다. 모델이 도구 호출을 돌려주면 도구를 실행해 결과를 되먹이고, 도구 호출이 없으면 답을 돌려주는 손 루프입니다.
- 검색을 했는데 근거가 한 번도 없었으면 모델의 답 대신 고정 안내 문장을 돌려줍니다. 「모른다고 답하라」를 모델 재량에만 맡기지 않습니다.

In [ ]:
llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
llm_tools = llm.bind_tools([faq_search])


def run_rag_agent(question: str, max_turn: int = 5) -> str:
    """검색 도구를 쥔 손 루프. 근거 없음 하드 가드와 반복 상한이 있다."""
    messages = [SystemMessage(content=SYSTEM), HumanMessage(content=question)]
    searched, found = False, False
    for _ in range(max_turn):
        res = llm_tools.invoke(messages)
        if not res.tool_calls:
            if searched and not found:
                return NO_EVIDENCE
            return res.content
        messages.append(res)
        for call in res.tool_calls:
            searched = True
            result = faq_search.invoke(call["args"])
            if not result.startswith("검색 결과 없음"):
                found = True
            print(f"  [검색 호출] {call['args']} → {result[:40].replace(chr(10), ' ')}…")
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
    return "반복 한도 초과"


for q in ["안녕하세요!", "자유이용권 환불 규정 알려 주세요", "파이썬 리스트 정렬은 어떻게 하나요?"]:
    print(f"[질문] {q}")
    print(f"  [답] {run_rag_agent(q)[:120]}")

### 서비스 단계 ③ — 앱·엔드포인트 등록 (요구사항 8)

- 서비스 파일은 노트북과 따로 도는 프로그램이므로 저장소를 다시 열고 도구·처리 함수를 파일 안에 그대로 둡니다.
- `@app.post("/ask")`가 주소와 함수를 잇습니다. 함수 안에서 처리 함수를 그대로 부르고, 반환한 값이 JSON 응답 본문이 됩니다.

In [ ]:
%%writefile app.py
# -*- coding: utf-8 -*-
"""구름월드 RAG 에이전트 서비스 — 서비스 표준 템플릿 구성 (실행: fastapi dev app.py --port 8031)."""

import os

from dotenv import find_dotenv, load_dotenv
from fastapi import FastAPI
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from pydantic import BaseModel

load_dotenv(find_dotenv(usecwd=True))

THRESHOLD = 1.5
NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."
SYSTEM = ("너는 시설 안내 담당자다. 인사말처럼 검색이 필요 없는 말에는 바로 답한다. "
          "그 밖의 모든 질문은 반드시 faq_search 도구로 근거를 먼저 찾고, 도구 결과에 있는 내용으로만 답한다. "
          "도구 결과가 '검색 결과 없음'이면 네가 아는 지식으로 답하지 말고 "
          f"'{NO_EVIDENCE}'라고만 답한다.")

emb = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(persist_directory=os.path.join(os.path.dirname(os.path.abspath(__file__)), "chroma_db"), embedding_function=emb)


def retrieve(query: str, k: int = 2):
    return db.similarity_search_with_score(query, k=k)


@tool(parse_docstring=True)
def faq_search(query: str) -> str:
    """시설 FAQ 문서에서 질문과 관련된 청크를 검색한다. 시설 이용, 요금, 환불, 운영 관련 질문에 쓴다.

    Args:
        query: 검색할 질의문. 결과가 없으면 표현을 바꿔 재검색할 수 있다.
    """
    hits = retrieve(query)
    good = [(d, s) for d, s in hits if s <= THRESHOLD]
    if not good:
        return f"검색 결과 없음 (최고 유사도 점수 {hits[0][1]:.2f}가 임계값 {THRESHOLD}를 넘음). 질의를 바꿔 다시 검색하거나, 모른다고 답하라."
    return "\n---\n".join(f"[score {s:.2f}] {d.page_content}" for d, s in good)


llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
llm_tools = llm.bind_tools([faq_search])


# ── ① 입출력 모양 선언
class AskIn(BaseModel):
    question: str


class AskOut(BaseModel):
    answer: str


# ── ② 처리 함수 구현
def run_rag_agent(question: str, max_turn: int = 5) -> str:
    messages = [SystemMessage(content=SYSTEM), HumanMessage(content=question)]
    searched, found = False, False
    for _ in range(max_turn):
        res = llm_tools.invoke(messages)
        if not res.tool_calls:
            if searched and not found:
                return NO_EVIDENCE
            return res.content
        messages.append(res)
        for call in res.tool_calls:
            searched = True
            result = faq_search.invoke(call["args"])
            if not result.startswith("검색 결과 없음"):
                found = True
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
    return "반복 한도 초과"


# ── ③ 앱 생성·엔드포인트 등록
app = FastAPI(title="구름월드 RAG 에이전트 서비스")


@app.get("/healthz")
def healthz() -> dict:
    return {"status": "ok"}


@app.post("/ask", response_model=AskOut)
def ask_endpoint(req: AskIn) -> AskOut:
    return AskOut(answer=run_rag_agent(req.question))


# ── ④ 기동·호출 확인: 터미널에서 `fastapi dev app.py --port 8031` 뒤 `python run_samples.py 8031`

### 서비스 단계 ④ — 기동·호출 확인 (요구사항 9)

- 노트북에서 개발 서버를 자식 프로세스로 띄우고, 살아 있는지 확인한 뒤 요청을 보내고, 끝나면 서버를 내립니다.
- 터미널에서는 `fastapi dev app.py --port 8031`로 띄우고 다른 터미널에서 `python run_samples.py 8031`로 같은 확인을 합니다. PowerShell의 `curl`은 별칭이므로 `curl.exe`나 `run_samples.py`를 씁니다.
- 포트 8031이 이미 쓰이고 있으면 노트북과 명령의 포트 숫자를 함께 바꿉니다.

In [ ]:
BASE = "http://127.0.0.1:8031"
import tempfile
LOG_PATH = os.path.join(tempfile.gettempdir(), "lec03_ex04_server_stderr.log")   # 기동 실패 원인을 보는 서버 오류 출력 파일
log = open(LOG_PATH, "w", encoding="utf-8")
server = subprocess.Popen([sys.executable, "-m", "fastapi", "dev", "app.py", "--port", "8031", "--no-reload"],
                          stdout=subprocess.DEVNULL, stderr=log,
                          env=dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1"))   # cp949 콘솔에서도 서버가 죽지 않게
started = False
try:
    for _ in range(60):                  # 2초 × 60회 = 최대 120초 대기
        try:
            if httpx.get(f"{BASE}/healthz", timeout=2).status_code == 200:
                started = True
                break
        except httpx.HTTPError:
            pass
        time.sleep(2)
    if not started:
        log.flush()
        tail = open(LOG_PATH, encoding="utf-8").read()[-1500:]
        raise RuntimeError(f"서버가 120초 안에 뜨지 않았습니다. 서버 오류 출력({LOG_PATH}): " + tail)
    print("[healthz] 200 ok")

    samples = json.load(open("samples.json", encoding="utf-8"))
    for i, payload in enumerate(samples, 1):
        r = httpx.post(f"{BASE}/ask", json=payload, timeout=120)
        print(f"[{i}] 요청: {payload['question']} | 상태: {r.status_code}")
        print(f"[{i}] 응답: {r.json()}")
finally:
    server.terminate()
    server.wait(timeout=10)
    log.close()
    if started:
        try:
            os.remove(LOG_PATH)
        except OSError:
            pass                     # 자식 프로세스가 아직 파일을 쥐고 있으면 임시 폴더에 그대로 둔다
    print("서버를 내렸습니다.")

## 7. 실행 결과 확인

위 실행 기록에서 다음 세 가지를 확인합니다.

1. 서비스 단계 ②에서 「안녕하세요!」는 `[검색 호출]` 줄 없이 즉답하고, 환불 질문은 `[검색 호출]` 한 줄 뒤 FAQ 근거로 답하며, 파이썬 질문은 `[검색 호출]` 뒤 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」로 끝납니다.
2. 서비스 단계 ④에서 `[healthz] 200 ok`가 찍히고, 세 요청의 상태가 모두 200이며 응답 JSON의 `answer`가 단계 ②의 답과 같은 성격입니다.
3. 실습 폴더에 `app.py`가 생겼고, 터미널에서 `fastapi dev app.py --port 8031`와 `python run_samples.py 8031`로 같은 응답을 받습니다.

세 가지가 모두 확인되면 완성입니다.